# Trading Analysis — SQL Queries

**Goal:** Load MT4 trade history into a SQLite database and answer key performance questions 
using SQL — mirroring how data analysts work with structured databases in practice.

**Approach:** Each query answers a specific business question about trading performance, 
written in pure SQL and visualised where relevant.

**Tools:** Python, SQLite3, pandas

## Setting Up the Database

In [1]:
import sqlite3
import pandas as pd

# load CSV and push into SQLite database
trades = pd.read_csv('../data/mt4_trade_history.csv')
conn = sqlite3.connect('../data/trades.db')
trades.to_sql('raw_trades', conn, if_exists='replace', index=False)

print("Database created successfully")
print(f"Rows loaded: {len(trades)}")

Database created successfully
Rows loaded: 135


In [2]:
# create a clean closed_trades view in SQL
query = """
CREATE TABLE IF NOT EXISTS closed_trades AS
SELECT 
    e.Position,
    e.Symbol,
    e.Date AS entry_date,
    e.Action AS direction,
    e.Price AS entry_price,
    e.Volume AS volume,
    o.exit_date,
    o.exit_reason,
    o.exit_price,
    o.gross_profit,
    o.commission,
    (o.gross_profit + o.commission) AS net_profit,
    ROUND((JULIANDAY(o.exit_date) - JULIANDAY(e.Date)) * 1440, 2) AS duration_mins,
    CASE WHEN (o.gross_profit + o.commission) > 0 THEN 1 ELSE 0 END AS winner
FROM raw_trades e
JOIN (
    SELECT 
        Position,
        MAX(Date) AS exit_date,
        MAX(Reason) AS exit_reason,
        MAX(Price) AS exit_price,
        SUM(Profit) AS gross_profit,
        SUM(Commission) AS commission
    FROM raw_trades
    WHERE Entry = 'Out'
    GROUP BY Position
) o ON e.Position = o.Position
WHERE e.Entry = 'In'
"""

# drop if exists and recreate
conn.execute("DROP TABLE IF EXISTS closed_trades")
conn.execute(query)
conn.commit()

# verify
result = pd.read_sql("SELECT COUNT(*) as total_trades FROM closed_trades", conn)
print(result)

   total_trades
0            65


## Query 1: Which day of the week is most profitable?

In [4]:
query = """
SELECT 
    CASE CAST(strftime('%w', entry_date) AS INTEGER)
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS day_of_week,
    COUNT(*) AS trades,
    ROUND(AVG(winner) * 100, 1) AS win_rate_pct,
    ROUND(SUM(net_profit), 2) AS net_pnl,
    ROUND(AVG(net_profit), 2) AS avg_profit
FROM closed_trades
GROUP BY strftime('%w', entry_date)
ORDER BY net_pnl DESC
"""

pd.read_sql(query, conn)

,day_of_week,trades,win_rate_pct,net_pnl,avg_profit
0,Thursday,18,27.8,172.04,9.56
1,Wednesday,10,40.0,94.28,9.43
2,Tuesday,4,75.0,63.38,15.85
3,Monday,17,35.3,-8.41,-0.49
4,Friday,16,25.0,-21.10,-1.32


**Insight:** Thursday and Wednesday are the most profitable days, generating $172 and $94 
respectively. Tuesday shows the highest win rate (75%) but only 4 trades — too small a 
sample to draw conclusions. Monday and Friday are both slightly negative, suggesting the 
strategy performs poorly at the start and end of the trading week. A simple filter — avoiding 
Monday and Friday entries — could meaningfully improve overall performance.

## Query 2: Does position size affect outcome?

In [5]:
query = """
SELECT 
    CASE 
        WHEN volume <= 0.3 THEN 'Small (≤0.3)'
        WHEN volume <= 0.6 THEN 'Medium (0.3-0.6)'
        ELSE 'Large (>0.6)'
    END AS position_size,
    COUNT(*) AS trades,
    ROUND(AVG(winner) * 100, 1) AS win_rate_pct,
    ROUND(SUM(net_profit), 2) AS net_pnl,
    ROUND(AVG(net_profit), 2) AS avg_profit
FROM closed_trades
GROUP BY position_size
ORDER BY net_pnl DESC
"""

pd.read_sql(query, conn)

,position_size,trades,win_rate_pct,net_pnl,avg_profit
0,Large (>0.6),21,42.9,362.90,17.28
1,Small (≤0.3),4,25.0,-18.09,-4.52
2,Medium (0.3-0.6),40,30.0,-44.62,-1.12


**Insight:** Larger position sizes (>0.6 lots) generate the most profit — $362.90 at a 42.9% 
win rate. Medium positions (0.3–0.6) are the most common (40 trades) but slightly negative 
overall. This suggests confidence in trade selection correlates with better outcomes — trades 
taken with larger size tend to be higher conviction setups that perform better. However, this 
could also reflect survivorship bias in a small dataset of 65 trades.

## Query 3: Longest winning and losing streaks

In [7]:
query = """
WITH streaks AS (
    SELECT 
        winner,
        ROW_NUMBER() OVER (ORDER BY entry_date) - 
        ROW_NUMBER() OVER (PARTITION BY winner ORDER BY entry_date) AS streak_group
    FROM closed_trades
),
streak_counts AS (
    SELECT 
        winner,
        COUNT(*) AS streak_length
    FROM streaks
    GROUP BY winner, streak_group
)
SELECT 
    CASE WHEN winner = 1 THEN 'Win Streak' ELSE 'Loss Streak' END AS streak_type,
    MAX(streak_length) AS max_streak
FROM streak_counts
GROUP BY winner
"""

pd.read_sql(query, conn)

,streak_type,max_streak
0,Loss Streak,11
1,Win Streak,4


**Insight:** The maximum losing streak is 11 consecutive losses, while the longest winning 
streak is 4. This is expected with a 33.8% win rate — long losing runs are mathematically 
normal and must be tolerated for the strategy to play out. A trader who stops after 5-6 
losses would never capture the winning trades that make the strategy profitable overall. 
Psychological resilience through drawdown periods is as important as the strategy itself.

## Query 4: Top 5 best and worst trades

In [8]:
query = """
SELECT position, symbol, direction, entry_date, exit_reason,
       volume, ROUND(net_profit, 2) AS net_profit
FROM closed_trades
ORDER BY net_profit DESC
LIMIT 5
"""
print("TOP 5 TRADES:")
print(pd.read_sql(query, conn).to_string(index=False))

query2 = """
SELECT position, symbol, direction, entry_date, exit_reason,
       volume, ROUND(net_profit, 2) AS net_profit
FROM closed_trades
ORDER BY net_profit ASC
LIMIT 5
"""
print("\nWORST 5 TRADES:")
print(pd.read_sql(query2, conn).to_string(index=False))

TOP 5 TRADES:
 Position Symbol direction          entry_date exit_reason  volume  net_profit
 97492048 EURUSD      Sell 2026-06-18T10:49:40          SL     0.7      251.20
 94054478 EURUSD      Sell 2026-05-27T16:34:07          SL     0.5      151.01
 94618283 EURUSD      Sell 2026-06-01T15:51:43          SL     0.5      111.00
 97261170 EURUSD      Sell 2026-06-17T12:26:01          SL     0.7      102.80
 95630704 GBPUSD      Sell 2026-06-05T18:03:30      Mobile     0.8      102.40

WORST 5 TRADES:
 Position Symbol direction          entry_date exit_reason  volume  net_profit
 96346916 EURUSD       Buy 2026-06-10T23:27:19          SL     0.6      -70.44
 94849855 GBPUSD       Buy 2026-06-02T14:53:11          SL     0.5      -65.50
 93630981 EURUSD       Buy 2026-05-25T09:49:41          SL     0.4      -56.00
 95654785 EURUSD       Buy 2026-06-05T20:00:18          SL     0.8      -55.20
 95628872 GBPUSD      Sell 2026-06-05T17:55:22          SL     0.8      -52.80


**Insight:** All 5 top trades are Sell direction on EURUSD, all exited via stop loss or mobile 
— confirming the short bias edge identified earlier. The largest single trade ($251.20) was a 
high conviction sell with 0.7 lots held overnight. 

The worst 5 trades are dominated by Buy direction — 4 of the 5 biggest losses are long trades, 
reinforcing that buying in this period was consistently the wrong call. The worst loss (-$70.44) 
was a Buy on EURUSD during a period of sustained downward pressure.

## Query 5: Weekly P&L progression

In [9]:
query = """
SELECT 
    strftime('%Y-%W', entry_date) AS week,
    COUNT(*) AS trades,
    ROUND(AVG(winner) * 100, 1) AS win_rate_pct,
    ROUND(SUM(net_profit), 2) AS weekly_pnl,
    ROUND(SUM(SUM(net_profit)) OVER (ORDER BY strftime('%Y-%W', entry_date)), 2) AS cumulative_pnl
FROM closed_trades
GROUP BY week
ORDER BY week
"""

pd.read_sql(query, conn)

,week,trades,win_rate_pct,weekly_pnl,cumulative_pnl
0,2026-20,6,33.3,-78.20,-78.20
1,2026-21,11,27.3,51.21,-26.99
2,2026-22,25,24.0,49.84,22.85
3,2026-23,7,57.1,-101.27,-78.42
4,2026-24,8,37.5,233.90,155.48
5,2026-25,6,33.3,44.61,200.09
6,2026-26,2,100.0,100.10,300.19


**Insight:** Performance improved significantly over the 6-week period. The first three weeks 
were volatile and largely breakeven (-$26.99 cumulative by end of week 21). Week 23 was the 
worst week (-$101.27) but was followed immediately by the best week ($233.90 in week 24). 
The final two weeks show increasing consistency — week 26 achieved 100% win rate on 2 trades. 
The upward trajectory suggests the strategy and execution are improving over time.

## SQL Analysis Summary

Key findings from database queries:

| Question | Finding |
|---|---|
| Best day | Thursday (+$172, 27.8% win rate) |
| Worst day | Friday (-$21, 25% win rate) |
| Best position size | Large >0.6 lots (+$362, 42.9% win rate) |
| Max losing streak | 11 consecutive losses |
| Max winning streak | 4 consecutive wins |
| Best trade | $251.20 (EURUSD Sell, 18 Jun) |
| Worst trade | -$70.44 (EURUSD Buy, 10 Jun) |
| Best week | Week 24 (+$233.90) |
| Trend | Improving — last 2 weeks profitable and consistent |